# OpenSearch-Prüfstand: Verbindung · Plugins · Rechte · Indizes · Ingest · Abfragen

**Zweck:** eine OpenSearch-Instanz *von außen* prüfen und dabei jede Anfrage sehen, die unser Code sonst versteckt. Erst gegen den lokalen Container (`opensearch-index/`, `make up`), dann – nur durch Umschalten von `PROFIL` – gegen die entfernte Instanz (HTTPS, Benutzer/Passwort).

**Was geprüft wird, in dieser Reihenfolge:**

1. Verbindung (`GET /`, `GET _cluster/health`)
2. Abhängigkeiten: Version, Plugins (`opensearch-knn` ist Pflicht), deutsche Analyse (`_analyze`)
3. Benutzer und Rechte (Security-Plugin, falls aktiv) + praktischer Test: temporären `knn_vector`-Index anlegen, schreiben, suchen, löschen
4. Die vier Indizes anlegen – Schema sichtbar, dann `osi` (`Indexer.bootstrap`)
5. `response.json` eines Handbuchs → Datensätze (was `osi` daraus extrahiert) → Bulk-Ingest
6. Die vier Suchkanäle als einzelne Requests (kNN, BM25, Identifier, Label)
7. Graph aus `…-documents` laden, 1 Hop, Belegstellen per `mget`
8. RRF: wie aus den Kanallisten die Top-Chunks werden

**Nicht enthalten:** LLM-Aufruf, Guardrail, Prompt. Dafür: `RETRIEVAL-FLOW.md`. Feld- und Abfragereferenz: `OPENSEARCH-INDEXES.md`.

**Präfix:** alle Indizes dieses Notebooks heißen `<PREFIX>-…` (Standard `nbtest`), damit nichts mit den produktiven `bhb-*`-Indizes kollidiert. Am Ende können sie gelöscht werden (`AUFRAEUMEN = True`).

Kernel: `Python (database_comparison)` (dort sind `opensearch_index`, `rag_retrieval`, `opensearch-py`, `pandas` installiert; siehe `database_comparison/README.md`).

## 1 · Variablen

Eine Zelle, zwei Profile. Geheimnisse (Passwort, API-Key) kommen aus `.env` (Vorlage `.env.example` im selben Ordner, gitignored), nie aus dem Notebook. **Umschalten = `PROFIL` ändern.**

| Variable | lokal | remote |
|---|---|---|
| `url` | `http://localhost:9200` (Security-Plugin aus) | `https://<host>:<port>` |
| `username`/`password` | – | Basic Auth (Security-Plugin) |
| `verify_certs`/`ca_certs` | – | Firmen-CA als PEM; `verify_certs=False` nur zum ersten Ausprobieren |
| `rrf_pipeline` | `True` | `False`, wenn der Benutzer kein `cluster:admin/opensearch/search/pipeline` darf (der Chat braucht die Pipeline nicht) |

In [1]:
import os, sys, json, time, logging
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env"); logging.basicConfig(level=logging.WARNING); logging.getLogger("opensearch").setLevel(logging.ERROR)   # 400/404 sind hier erwartete Antworten
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

PROFIL = "lokal"                                   # "lokal" | "remote"
PROFILE = {
    "lokal":  {"url": "http://localhost:9200", "username": None, "password": None,
               "verify_certs": True, "ca_certs": None, "rrf_pipeline": True},
    "remote": {"url": os.environ.get("OS_REMOTE_URL", "https://opensearch.intern:9200"),
               "username": os.environ.get("OS_REMOTE_USER"), "password": os.environ.get("OS_REMOTE_PASSWORD"),
               "verify_certs": os.environ.get("OS_REMOTE_VERIFY", "true").lower() == "true",
               "ca_certs": os.environ.get("OS_REMOTE_CA") or None, "rrf_pipeline": False},
}
VERBINDUNG = PROFILE[PROFIL]

PREFIX        = "nbtest"                                            # Indizes nbtest-chunks / -nodes / -documents / -manifest
RUN_DIR       = "../out/remote-zsd"                              # ein Handbuch: response.json von docling-graph (ZSD)
ONTOLOGY_PATH = "../user-manual-books/handbuch_daten/Ontologie/ontology.yaml"
EMB_MODEL, EMB_DIM, EMB_TEXT_PREFIX = "bge-m3", 1024, ""             # was die Chunk-Vektoren erzeugt hat (response.json nennt es nicht)

# Frage-Embedding für den kNN-Kanal (OpenAI-kompatibel, /v1/embeddings). Leer lassen → Ersatz: gespeicherter Vektor eines Chunks (§7).
EMB_BASE_URL  = os.environ.get("RAG__EMBEDDING__BASE_URL")          # z. B. http://localhost:4000/v1
EMB_API_KEY   = os.environ.get("RAG__EMBEDDING__API_KEY")

FRAGE      = "Wer ist verantwortlich für SOP-ZSD-06?"               # Identifier + Label im ZSD-Handbuch
K_CHANNEL  = 20                                                     # Treffer je Kanal (wie rag_retrieval: k_per_channel)
FINAL_K    = 10                                                     # Endliste (final_k)
AUFRAEUMEN = False                                                  # True: am Ende alle <PREFIX>-* Indizes + Pipeline löschen

print(f"Profil {PROFIL}: {VERBINDUNG['url']}  Benutzer: {VERBINDUNG['username'] or '(keiner)'}  Präfix: {PREFIX}")

Profil lokal: http://localhost:9200  Benutzer: (keiner)  Präfix: nbtest


## 2 · Verbindung

`opensearch_index.client.make_client` baut den `opensearch-py`-Client: `OpenSearch(hosts=[url], http_auth=(user, pw), use_ssl=True, verify_certs=…, ca_certs=…)`. Darunter ist alles HTTP + JSON: jede Methode des Clients (`client.search`, `client.indices.create`, …) ist ein REST-Aufruf, und `client.transport.perform_request(method, path, body)` zeigt ihn roh. Der Helfer `req` unten macht genau das und fängt Fehler ab (Status 401/403 = Rechte, 404 = Endpunkt oder Plugin fehlt).

In [2]:
from opensearch_index.settings import Settings as OsiSettings, OpenSearchSettings, IndexSettings, EmbeddingSettings, OntologySettings
from opensearch_index.client import make_client, cluster_info
from opensearch_index.mappings import IndexNames

settings = OsiSettings(                                             # explizit, keine versteckte .env/config.yaml-Magie
    opensearch=OpenSearchSettings(url=VERBINDUNG["url"], username=VERBINDUNG["username"], password=VERBINDUNG["password"],
                                  verify_certs=VERBINDUNG["verify_certs"], ca_certs=VERBINDUNG["ca_certs"]),
    index=IndexSettings(prefix=PREFIX, rrf_pipeline=VERBINDUNG["rrf_pipeline"]),       # shards=1, replicas=0, m=16, ef_construction=128
    embedding=EmbeddingSettings(model=EMB_MODEL, dim=EMB_DIM, text_prefix=EMB_TEXT_PREFIX),
    ontology=OntologySettings(path=ONTOLOGY_PATH),
)
client = make_client(settings.opensearch)
NAMES = IndexNames(PREFIX)

def req(method, path, body=None, params=None):
    "Roher REST-Aufruf; gibt (status, body) zurück statt zu werfen."
    try:
        return 200, client.transport.perform_request(method, path, body=body, params=params)
    except Exception as exc:
        return getattr(exc, "status_code", "?"), str(getattr(exc, "info", None) or exc)[:300]

status, root = req("GET", "/")
assert status == 200, f"keine Verbindung: {root}"
status, health = req("GET", "/_cluster/health")
print(f"GET /                 → {root['version']['distribution']} {root['version']['number']}, Cluster „{root['cluster_name']}“, Lucene {root['version']['lucene_version']}")
print(f"GET /_cluster/health  → Status {health['status']}, Knoten {health['number_of_nodes']}, Shards aktiv {health['active_shards']}")

GET /                 → opensearch 3.8.0, Cluster „bhb“, Lucene 10.5.0
GET /_cluster/health  → Status green, Knoten 1, Shards aktiv 28


## 3 · Abhängigkeiten: Plugins und Analyse

Was die vier Indizes vom Server brauchen:

| Komponente | Wofür | Prüfung |
|---|---|---|
| `opensearch-knn` (Plugin) | Feldtyp `knn_vector`, Abfrage `knn`, Engine `lucene` | `GET _cat/plugins` |
| BM25 + Modul `analysis-common` (im Kern) | Analyzer `de_text`: Standard-Tokenizer, `lowercase`, Stoppwörter `_german_`, `german_normalization`, Stemmer `light_german` | `POST _analyze` mit der Definition inline |
| Version ≥ 2.19 | nur für die Search-Pipeline `rrf` (`osi search --hybrid`); der Chat fusioniert selbst | Versionsnummer |
| `opensearch-security` (Plugin) | Benutzer, TLS – auf dem lokalen Container abgeschaltet | §4 |

In [3]:
info = cluster_info(client)                                         # GET / + GET _cat/plugins?format=json
status, plugins = req("GET", "/_cat/plugins", params={"format": "json", "h": "name,component,version"})
display(pd.DataFrame(plugins).drop_duplicates("component").reset_index(drop=True))
print(f"k-NN-Plugin: {'JA' if info.has_knn else 'FEHLT – knn_vector/knn-Abfrage unmöglich'}   ·   Version {info.version} → RRF-Pipeline {'möglich' if info.supports_rrf else 'nicht möglich (< 2.19)'}")
status, knn_stats = req("GET", "/_plugins/_knn/stats", params={"stat": "indices_in_cache,graph_memory_usage"})
print("GET /_plugins/_knn/stats →", status, (json.dumps(knn_stats)[:160] if status == 200 else knn_stats))

# Deutsche Analyse ohne Index: dieselben Filter wie in mappings.analysis_settings()
probe = "Die Zertifikate der Sicherheitsdienste müssen erneuert werden: SOP-ZSD-06, Dr. Annika Reuß"
status, ana = req("POST", "/_analyze", body={"tokenizer": "standard",
    "filter": ["lowercase", {"type": "stop", "stopwords": "_german_"}, "german_normalization", {"type": "stemmer", "language": "light_german"}],
    "text": probe})
tokens = [t["token"] for t in ana["tokens"]] if status == 200 else ana
print(f"POST /_analyze (de_text inline) → {status}: {tokens}")
print("→ Beachte: „SOP-ZSD-06“ wird zu", [t for t in tokens if t in ("sop", "zsd", "06")], "– deshalb gibt es den Identifier-Kanal (§7).")

,name,component,version
0,opensearch-1,opensearch-alerting,3.8.0.0
1,opensearch-1,opensearch-anomaly-detection,3.8.0.0
2,opensearch-1,opensearch-asynchronous-search,3.8.0.0
3,opensearch-1,opensearch-cross-cluster-replication,3.8.0.0
4,opensearch-1,opensearch-custom-codecs,3.8.0.0
5,opensearch-1,opensearch-flow-framework,3.8.0.0
6,opensearch-1,opensearch-geospatial,3.8.0.0
7,opensearch-1,opensearch-index-management,3.8.0.0
8,opensearch-1,opensearch-job-scheduler,3.8.0.0
9,opensearch-1,opensearch-knn,3.8.0.0


k-NN-Plugin: JA   ·   Version 3.8.0 → RRF-Pipeline möglich
GET /_plugins/_knn/stats → 200 {"_nodes": {"total": 1, "successful": 1, "failed": 0}, "cluster_name": "bhb", "nodes": {"gjbFChf2SGaemCbdWgM9YA": {"indices_in_cache": {}, "graph_memory_usage":
POST /_analyze (de_text inline) → 200: ['zertifikat', 'sicherheitsdien', 'muss', 'erneuert', 'sop', 'zsd', '06', 'dr', 'annika', 'reuss']
→ Beachte: „SOP-ZSD-06“ wird zu ['sop', 'zsd', '06'] – deshalb gibt es den Identifier-Kanal (§7).


## 4 · Benutzer und Rechte

Mit aktivem Security-Plugin sagt `GET _plugins/_security/authinfo`, **wer** wir sind (Benutzer, Rollen, Backend-Rollen); `GET _plugins/_security/api/roles/<rolle>` zeigt die **Rechte** der Rolle (Cluster-Aktionen, Index-Muster) – das darf oft nur ein Admin lesen (403), dann hilft der praktische Test darunter.

**Praktischer Test** (`opensearch_index.client.verify_knn_roundtrip`): legt `<PREFIX>-verify-<zufall>` mit genau der produktiven `knn_vector`-Definition an (dim 1024, `cosinesimil`, HNSW/`lucene`), schreibt 3 Einheitsvektoren, sucht den ersten mit `k=1`, erwartet ihn zurück, löscht den Index. Besteht er, hat der Benutzer `indices:admin/create`, `indices:data/write`, `indices:data/read/search`, `indices:admin/delete` auf dem Muster – und das k-NN-Plugin funktioniert in dieser Dimension.

In [4]:
from opensearch_index.client import verify_knn_roundtrip

status, who = req("GET", "/_plugins/_security/authinfo")
if status == 200:
    print(f"Benutzer: {who['user_name']}  Rollen: {who['roles']}  Backend-Rollen: {who['backend_roles']}")
    for role in who["roles"]:
        st, r = req("GET", f"/_plugins/_security/api/roles/{role}")
        if st == 200:
            print(f"  Rolle {role}: cluster={r[role].get('cluster_permissions')} index={[(p['index_patterns'], p['allowed_actions']) for p in r[role].get('index_permissions', [])]}")
        else:
            print(f"  Rolle {role}: Rechte nicht lesbar ({st}) → praktischer Test unten")
else:
    print(f"Security-Plugin nicht aktiv oder nicht erreichbar ({status}): keine Benutzer, keine Rechteprüfung (lokaler Container: DISABLE_SECURITY_PLUGIN=true)")

t0 = time.perf_counter()
rt = verify_knn_roundtrip(client, dim=EMB_DIM, names=NAMES)         # anlegen → 3 Vektoren → knn k=1 → löschen
print(f"\nkNN-Roundtrip auf {rt['index']}: {'OK' if rt.get('hit') == 'v0' else 'FEHLGESCHLAGEN'} ({time.perf_counter() - t0:.1f} s) → {json.dumps({k: v for k, v in rt.items() if k != 'index'})[:200]}")

Security-Plugin nicht aktiv oder nicht erreichbar (400): keine Benutzer, keine Rechteprüfung (lokaler Container: DISABLE_SECURITY_PLUGIN=true)



kNN-Roundtrip auf nbtest-verify-9967fa51: OK (0.5 s) → {"dim": 1024, "hit": "v0", "score": 1.0}


## 5 · Die vier Indizes anlegen

Was `osi bootstrap` tut (`opensearch-index/src/opensearch_index/indexer.py` → `Indexer.bootstrap`):

1. `mappings.index_bodies()` liefert je Index den Body für `PUT /<index>`: `settings` (Shards, Replikate, Analyzer `de_text`, bei Chunks `index.knn: true`) + `mappings` (`dynamic: strict`) + `aliases`. Zahlen kommen aus `config.yaml` bzw. unseren `IndexSettings`/`EmbeddingSettings` oben.
2. Für jeden Alias: existiert `<alias>-v1`? Nein → `PUT /<alias>-v1` mit dem Body (Alias wird im Body mit angelegt). Ja → nur sicherstellen, dass der Alias hängt.
3. `check_indices()`: `GET /<alias>/_mapping` und `_meta.mapping_version`/`embedding_dim` mit der Konfiguration vergleichen – ein fremder Index wird nicht beschrieben.
4. Optional `PUT /_search/pipeline/<prefix>-rrf` (nur wenn `rrf_pipeline=True` und Version ≥ 2.19).

Zuerst die Bodies als Text – das *ist* das Schema.

In [5]:
from opensearch_index.mappings import index_bodies, MAPPING_VERSION, KNN_ENGINE, SPACE_TYPE

bodies = index_bodies(NAMES, settings.embedding, settings.index)
def feldtypen(mapping):
    rows = []
    for name, p in mapping["properties"].items():
        t = p.get("type", "object")
        if t == "knn_vector": t += f" dim={p['dimension']} {p['space_type']} {p['method']['name']}/{p['method']['engine']} m={p['method']['parameters']['m']} ef={p['method']['parameters']['ef_construction']}"
        if p.get("analyzer"): t += f"({p['analyzer']})"
        if p.get("normalizer"): t += f" normalizer={p['normalizer']}"
        if p.get("fields"): t += " + .keyword"
        if p.get("index") is False: t += " (nicht indexiert)"
        if p.get("enabled") is False: t += " (disabled)"
        if "properties" in p: t = "object{" + ", ".join(p["properties"]) + "}"
        rows.append({"Feld": name, "Typ": t})
    return pd.DataFrame(rows)

print("Settings des Chunk-Index (identisch für alle vier bis auf index.knn):")
print(json.dumps(bodies[NAMES.chunks]["settings"], ensure_ascii=False, indent=1)[:900])
print("\n_meta des Chunk-Mappings:", bodies[NAMES.chunks]["mappings"]["_meta"])
for alias, body in bodies.items():
    display(Markdown(f"**`PUT /{NAMES.physical(alias)}`** → Alias `{alias}` · {len(body['mappings']['properties'])} Felder · `dynamic: {body['mappings']['dynamic']}`"))
    display(feldtypen(body["mappings"]))

Settings des Chunk-Index (identisch für alle vier bis auf index.knn):
{
 "index": {
  "number_of_shards": 1,
  "number_of_replicas": 0,
  "knn": true
 },
 "analysis": {
  "filter": {
   "de_stop": {
    "type": "stop",
    "stopwords": "_german_"
   },
   "de_stem": {
    "type": "stemmer",
    "language": "light_german"
   }
  },
  "normalizer": {
   "lc": {
    "type": "custom",
    "filter": [
     "lowercase",
     "trim"
    ]
   }
  },
  "analyzer": {
   "de_text": {
    "type": "custom",
    "tokenizer": "standard",
    "filter": [
     "lowercase",
     "de_stop",
     "german_normalization",
     "de_stem"
    ]
   }
  }
 }
}

_meta des Chunk-Mappings: {'mapping_version': 1, 'embedding_dim': 1024, 'embedding_model': 'bge-m3', 'knn_engine': 'lucene', 'space_type': 'cosinesimil'}


**`PUT /nbtest-chunks-v1`** → Alias `nbtest-chunks` · 22 Felder · `dynamic: strict`

,Feld,Typ
0,doc_id,keyword
1,doc_sha256,keyword
2,run_id,keyword
3,indexed_at,date
4,chunk_id,keyword
5,text,text(de_text)
6,body_text,text(de_text)
7,heading_breadcrumb,text(de_text) + .keyword
8,heading_level,integer
9,kind,keyword


**`PUT /nbtest-nodes-v1`** → Alias `nbtest-nodes` · 20 Felder · `dynamic: strict`

,Feld,Typ
0,doc_id,keyword
1,doc_sha256,keyword
2,run_id,keyword
3,indexed_at,date
4,node_id,keyword
5,scope,keyword
6,type,keyword
7,label,text(standard) + .keyword
8,aliases,text(standard) + .keyword
9,identity,keyword


**`PUT /nbtest-documents-v1`** → Alias `nbtest-documents` · 29 Felder · `dynamic: strict`

,Feld,Typ
0,doc_id,keyword
1,doc_sha256,keyword
2,run_id,keyword
3,indexed_at,date
4,doc_id_source,keyword
5,scope,keyword
6,name,keyword
7,format,keyword
8,pages,integer
9,tables,integer


**`PUT /nbtest-manifest-v1`** → Alias `nbtest-manifest` · 17 Felder · `dynamic: strict`

,Feld,Typ
0,doc_id,keyword
1,status,keyword
2,run_id,keyword
3,previous_run_id,keyword
4,doc_sha256,keyword
5,doc_name,keyword
6,doc_id_source,keyword
7,owner,keyword
8,started_at,date
9,finished_at,date


In [6]:
from opensearch_index.indexer import Indexer
from opensearch_index.ontology import load_ontology

ontology = load_ontology(ONTOLOGY_PATH)                              # Regexe für Identifier, Identitätsregeln, meta.documents
indexer = Indexer(client, settings, ontology)
report = indexer.bootstrap()                                         # idempotent: vorhandene Indizes bleiben
print("bootstrap →", json.dumps(report))

status, cat = req("GET", f"/_cat/indices/{PREFIX}-*", params={"format": "json", "h": "index,health,status,docs.count"})
status, aliases = req("GET", f"/_cat/aliases/{PREFIX}-*", params={"format": "json", "h": "alias,index"})
display(pd.DataFrame(cat).merge(pd.DataFrame(aliases), on="index", how="left"))

status, m = req("GET", f"/{NAMES.chunks}/_mapping")                 # was der Server wirklich gespeichert hat
phys = next(iter(m)); props = m[phys]["mappings"]["properties"]
print(f"GET /{NAMES.chunks}/_mapping → physisch {phys}; embedding = {json.dumps(props['embedding'])}")

bootstrap → {"created": ["nbtest-chunks-v1", "nbtest-nodes-v1", "nbtest-documents-v1", "nbtest-manifest-v1"], "existing": [], "pipeline": "nbtest-rrf", "dry_run": false}


,index,health,status,docs.count,alias
0,nbtest-nodes-v1,green,open,0,nbtest-nodes
1,nbtest-manifest-v1,green,open,0,nbtest-manifest
2,nbtest-chunks-v1,green,open,0,nbtest-chunks
3,nbtest-documents-v1,green,open,0,nbtest-documents


GET /nbtest-chunks/_mapping → physisch nbtest-chunks-v1; embedding = {"type": "knn_vector", "dimension": 1024, "method": {"engine": "lucene", "space_type": "cosinesimil", "name": "hnsw", "parameters": {"ef_construction": 128, "m": 16}}, "space_type": "cosinesimil"}


## 6 · `response.json` → Datensätze → Ingest

Was `osi ingest <ordner>` tut, Schritt für Schritt (alles in `opensearch_index/`):

| Schritt | Funktion | Was passiert |
|---|---|---|
| Laden | `loader.load_run` | `response.json` lesen, gegen `schemas.ProcessResponse` validieren (Pydantic): `document`, `markdown`, `chunks[]` (mit `embedding`), `graph{nodes, edges}` |
| Planen | `Indexer.plan` | `doc_id` auflösen (`identity.resolve_doc_id`: Override → Ontologie `meta.documents` nach Dateiname → Regex auf Seite 1 → `DOCUMENTS`-Kante); `run_id` = Hash über Inhalt + Kanten-IDs + Ontologie + Embedding-Modell + Mapping-Version; Manifest lesen → `ingest` / `noop` / `replace` |
| Umwandeln | `transform.build_batch` | **Chunks:** Felder aus `c.*` + `identifiers` (Ontologie-Regexe über `text`+`caption`) + `node_ids`/`node_labels`/`node_types`/`edge_ids` (Rückwärtsindex aus `graph.*.provenance.chunk_ids`); Vektor-Dimension prüfen. **Nodes:** je Knoten ein Datensatz mit `identity`, `degree`, `identifiers`. **Document:** Metadaten aus dem `Document`-Knoten + `graph` als Blob (Kanten bekommen `id` = Hash). |
| Schreiben | `Indexer.ingest` | Manifest-Sperre (`PUT …-manifest/_doc/<doc_id>?op_type=create`, Status `indexing`) → `POST /_bulk` (`transform.bulk_actions`: `_id` = `chunk_id` / `sha12:node_id` / `doc_id`) → `_refresh` → Zählung je Index gegen den Plan → Sweep älterer Läufe (`delete_by_query doc_id ∧ ¬run_id`) → Manifest `active` |

In [7]:
from opensearch_index.loader import load_run
from opensearch_index.transform import build_batch, bulk_actions

resp = load_run(RUN_DIR)
print(f"load_run: {resp.document.name} · sha256 {resp.document.sha256[:12]}… · {resp.document.pages} Seiten · "
      f"{len(resp.chunks)} Chunks (Vektoren: {sum(1 for c in resp.chunks if c.embedding)} × {len(resp.chunks[0].embedding or [])}) · "
      f"Graph {len(resp.graph.nodes)} Knoten / {len(resp.graph.edges)} Kanten · degraded {resp.degraded.model_dump()}")

plan = indexer.plan(resp)
print(f"\nplan: doc_id={plan.doc_id} (Quelle: {plan.doc_id_source}) · run_id={plan.run_id[:16]}… · Aktion={plan.action} · Gründe={plan.reasons} · Warnungen={plan.warnings}")

batch = build_batch(resp, doc_id=plan.doc_id, doc_id_source=plan.doc_id_source, run_id=plan.run_id, ontology=ontology,
                    embedding_model=EMB_MODEL, embedding_dim=EMB_DIM, embedding_text_prefix=EMB_TEXT_PREFIX, indexed_at="2026-01-01T00:00:00Z")
print(f"build_batch: {batch.counts}")

c = next(c for c in batch.chunks if "SOP-ZSD-06" in (c.get("identifiers") or []))   # ein Chunk, der die Frage trifft
print("\nEin Chunk-Datensatz (ohne embedding/bboxes):")
print(json.dumps({k: (v[:120] + "…" if isinstance(v, str) and len(v) > 120 else v) for k, v in c.items() if k not in ("embedding", "bboxes")}, ensure_ascii=False, indent=1))
n = next(n for n in batch.nodes if n["node_id"] in c["node_ids"] and n["type"] == "Person")
print("\nEin Node-Datensatz (ohne provenance):")
print(json.dumps({k: v for k, v in n.items() if k != "provenance"}, ensure_ascii=False, indent=1))
d = batch.document
print("\nDocument-Datensatz (Schlüssel):", sorted(d), f"\n  title={d['title']!r} version={d['version']!r} graph: {len(d['graph']['nodes'])} Knoten, {len(d['graph']['edges'])} Kanten, erste Kante mit id={d['graph']['edges'][0]['id']}")
print("\nErste Bulk-Aktionen (_index/_id):", [(a["_index"], a["_id"]) for a in list(bulk_actions(batch, NAMES))[:3]], "… insgesamt", len(batch.chunks) + len(batch.nodes) + 1)

load_run: Betriebshandbuch_ZSD.pdf · sha256 2bb722f565a0… · 29 Seiten · 112 Chunks (Vektoren: 112 × 1024) · Graph 250 Knoten / 154 Kanten · degraded {'vlm': False, 'embeddings': False, 'graph': False}



plan: doc_id=BHB-PLT-0007 (Quelle: ontology_file) · run_id=80638e76f8a15e34… · Aktion=ingest · Gründe=['no manifest for this doc_id'] · Warnungen=[]
build_batch: {'chunks': 112, 'nodes': 250, 'edges': 154, 'documents': 1}

Ein Chunk-Datensatz (ohne embedding/bboxes):
{
 "doc_id": "BHB-PLT-0007",
 "doc_sha256": "2bb722f565a0ff49ccb42790a6ada683d86cef0486c8a1da8b55f5265c845c72",
 "run_id": "80638e76f8a15e34542571332d5c91fc8c3d5c850e063a4c79311bc645b3c9ad",
 "indexed_at": "2026-01-01T00:00:00Z",
 "chunk_id": "2bb722f565a0-0055",
 "text": "5 Betrieb\n5.5 Standard Operating Procedures\nTabelle 7: Standard Operating Procedures\n\n| SOP | Titel | Auslöser | Verantw…",
 "body_text": "Tabelle 7: Standard Operating Procedures\n\n| SOP | Titel | Auslöser | Verantwortlich | Dauer | Abstimmung |\n| - | - | - |…",
 "heading_breadcrumb": [
  "5 Betrieb",
  "5.5 Standard Operating Procedures"
 ],
 "heading_level": 2,
 "kind": "table",
 "caption": "Tabelle 7: Standard Operating Procedures",
 "page_num

In [8]:
rep = indexer.ingest(plan, resp)                                     # Sperre → _bulk → _refresh → Zählung → Sweep → Manifest active
print(f"ingest → {rep.status} in {rep.duration_s} s · geschrieben {rep.counts} · weggeräumt {rep.swept}")
rows = []
for alias in NAMES.aliases:
    st, r = req("GET", f"/{alias}/_count", body={"query": {"term": {"doc_id": plan.doc_id}}})
    rows.append({"Index": alias, f"Datensätze doc_id={plan.doc_id}": r["count"] if st == 200 else r})
display(pd.DataFrame(rows))
st, mf = req("GET", f"/{NAMES.manifest}/_doc/{plan.doc_id}")
print("Manifest:", json.dumps({k: mf["_source"].get(k) for k in ("status", "run_id", "doc_name", "owner", "finished_at", "counts")}, ensure_ascii=False))

ingest → active in 1.268 s · geschrieben {'chunks': 112, 'nodes': 250, 'edges': 154, 'documents': 1} · weggeräumt {'chunks': 0, 'nodes': 0, 'documents': 0, 'edges': 0}


,Index,Datensätze doc_id=BHB-PLT-0007
0,nbtest-chunks,112
1,nbtest-nodes,250
2,nbtest-documents,1
3,nbtest-manifest,1


Manifest: {"status": "active", "run_id": "80638e76f8a15e34542571332d5c91fc8c3d5c850e063a4c79311bc645b3c9ad", "doc_name": "Betriebshandbuch_ZSD.pdf", "owner": "javi-pc:14271", "finished_at": "2026-10-27T15:02:15Z", "counts": {"chunks": 112, "nodes": 250, "edges": 154, "documents": 1}}


## 7 · Die vier Suchkanäle

Alles vor den Abfragen ist reines Python (`rag_retrieval.query.analyze_question`): **Identifier** = Ontologie-Regexe über die Frage; **Label-Begriffe** = Knotennamen aus dem Graphen, die als n-Gramme in der Frage stehen (dafür wird der Graph in §8 geladen – hier vorgezogen). Der **Vektor** der Frage kommt vom Embedding-Modell; ohne Endpunkt nehmen wir ersatzweise den gespeicherten Vektor eines Chunks, der die Frage trifft (`GET …-chunks/_doc/<id>?_source_includes=embedding`) – dann ist kNN „ähnliche Chunks zu diesem Chunk“ und Rang 1 der Chunk selbst.

Die Request-Bodies baut `rag_retrieval.search` (`knn_body`, `bm25_body`, `identifier_body`, `label_body`); im Chat gehen alle vier in **einem** `POST /_msearch`. Hier werden sie einzeln gezeigt und dann genau so gesendet.

In [9]:
from rag_retrieval import search as rs
from rag_retrieval.query import analyze_question
from rag_retrieval.graph import GraphStore
from rag_retrieval.fusion import RawHit

docs = rs.fetch_documents(client, NAMES.documents, include_graph=True)   # §8 erklärt diesen Aufruf; hier nur für die Label-Analyse
graph = GraphStore.from_documents(docs)
plan_q = analyze_question(FRAGE, regexes=ontology.regexes, graph=graph)
print(f"Frage: {FRAGE}\n  Identifier (Regex): {plan_q.identifiers}\n  Label-Begriffe (Graph): {plan_q.label_terms}\n  Startknoten: {[f'{graph.label(n)} ({graph.type_of(n)})' for n in plan_q.start_nodes]}")

vector, vektor_quelle = None, None
if EMB_BASE_URL:
    from rag_retrieval.embed import QuestionEmbedder
    from rag_retrieval.settings import EmbeddingSettings as RagEmb
    try:
        vector = QuestionEmbedder.from_settings(RagEmb(base_url=EMB_BASE_URL, api_key=EMB_API_KEY, model=EMB_MODEL, dim=EMB_DIM)).embed(FRAGE)
        vektor_quelle = f"POST {EMB_BASE_URL}/embeddings model={EMB_MODEL}"
    except Exception as exc:
        print("Embedding-Endpunkt nicht nutzbar:", str(exc)[:160])
if vector is None:
    st, r = req("GET", f"/{NAMES.chunks}/_search", body={"size": 1, "_source": ["embedding"], "query": {"term": {"identifiers": plan_q.identifiers[0]}}})
    hit = r["hits"]["hits"][0]; vector = hit["_source"]["embedding"]; vektor_quelle = f"Ersatz: gespeicherter Vektor von {hit['_id']}"
print(f"  Vektor: {len(vector)} dim, {[round(x, 4) for x in vector[:4]]} … ({vektor_quelle})")

bodies = {"knn": rs.knn_body(vector, K_CHANNEL), "bm25": rs.bm25_body(FRAGE, K_CHANNEL)}
if plan_q.identifiers: bodies["identifier"] = rs.identifier_body(plan_q.identifiers, K_CHANNEL)
if plan_q.label_terms:  bodies["label"] = rs.label_body(plan_q.label_terms, K_CHANNEL)
print("\nRequest-Bodies (POST /" + NAMES.chunks + "/_search):")
for ch, b in bodies.items():
    shown = json.loads(json.dumps(b))
    if ch == "knn": shown["query"]["knn"]["embedding"]["vector"] = shown["query"]["knn"]["embedding"]["vector"][:2] + ["…"]
    print(f"  {ch:10} {json.dumps(shown, ensure_ascii=False)}")

Frage: Wer ist verantwortlich für SOP-ZSD-06?
  Identifier (Regex): ['SOP-ZSD-06']
  Label-Begriffe (Graph): ['sop-zsd-06', 'sop-zsd-01', 'client-secret rotieren (sop-zsd-03)', 'sop-zsd-05', 'sop-zsd-02']
  Startknoten: ['SOP-ZSD-06 (Procedure)']


  Vektor: 1024 dim, [-0.0316, -0.0484, -0.0415, -0.0429] … (POST http://localhost:4000/v1/embeddings model=bge-m3)

Request-Bodies (POST /nbtest-chunks/_search):
  knn        {"size": 20, "_source": {"excludes": ["embedding", "bboxes"]}, "query": {"knn": {"embedding": {"vector": [-0.031647436, -0.04840559, "…"], "k": 20}}}}
  bm25       {"size": 20, "_source": {"excludes": ["embedding", "bboxes"]}, "query": {"multi_match": {"query": "Wer ist verantwortlich für SOP-ZSD-06?", "fields": ["text^2", "body_text", "caption"]}}}
  identifier {"size": 20, "_source": {"excludes": ["embedding", "bboxes"]}, "query": {"bool": {"should": [{"term": {"identifiers": "SOP-ZSD-06"}}], "minimum_should_match": 1}}}
  label      {"size": 20, "_source": {"excludes": ["embedding", "bboxes"]}, "query": {"bool": {"should": [{"term": {"node_labels": "sop-zsd-06"}}, {"term": {"node_labels": "sop-zsd-01"}}, {"term": {"node_labels": "client-secret rotieren (sop-zsd-03)"}}, {"term": {"node_labels": "sop-zsd-05"}}, {

In [10]:
lists, rows, took = {}, [], {}
for ch, body in bodies.items():
    st, r = req("POST", f"/{NAMES.chunks}/_search", body=body)      # einzeln, damit jeder Request für sich sichtbar ist (Chat: ein _msearch)
    assert st == 200, (ch, r)
    took[ch] = r["took"]
    lists[ch] = [RawHit(chunk_id=h["_id"], score=h["_score"], source=h["_source"]) for h in r["hits"]["hits"]]
    for i, h in enumerate(lists[ch][:6], start=1):
        s = h.source
        rows.append({"Kanal": ch, "Rang": i, "chunk_id": h.chunk_id, "Score": round(h.score, 4), "Seite": ",".join(map(str, s["page_numbers"])), "kind": s["kind"],
                     "Wo": (" › ".join(s["heading_breadcrumb"][-2:]) + (" · " + s["caption"] if s.get("caption") else ""))[:70],
                     "Identifier": ",".join(s.get("identifiers") or [])[:40]})
print("Treffer je Kanal:", {ch: len(l) for ch, l in lists.items()}, "· took ms:", took)
print("Scores sind je Kanal anders skaliert (kNN ≤ 1, BM25 ~ 5–20, term = IDF-Konstante) – die Fusion nutzt nur Ränge (§9).")
display(pd.DataFrame(rows))

Treffer je Kanal: {'knn': 20, 'bm25': 20, 'identifier': 2, 'label': 15} · took ms: {'knn': 10, 'bm25': 12, 'identifier': 8, 'label': 9}
Scores sind je Kanal anders skaliert (kNN ≤ 1, BM25 ~ 5–20, term = IDF-Konstante) – die Fusion nutzt nur Ränge (§9).


,Kanal,Rang,chunk_id,Score,Seite,kind,Wo,Identifier
0,knn,1,2bb722f565a0-0055,0.7758,17,table,5 Betrieb › 5.5 Standard Operating Procedures · Tabelle 7: Standard Op,"SOP-CAAS-06,SOP-ZSD-01,SOP-ZSD-02,SOP-ZS"
1,knn,2,2bb722f565a0-0050,0.7674,15,text,5 Betrieb › 5.4 Zertifikatslebenszyklus,"SOP-ZSD-01,SOP-ZSD-04"
2,knn,3,2bb722f565a0-0002,0.7645,1,table,ZSD - Zentrale Sicherheits dienste,BHB-PLT-0007
3,knn,4,2bb722f565a0-0044,0.7518,"13,14",text,5.3 Standardabläufe › 5.3.1 Keycloak rollierend neu starten (SOP-ZSD-0,"SOP-ZSD-02,ZSDSUP-0252,iam-p01,iam-p02,i"
4,knn,5,2bb722f565a0-0074,0.7504,23,table,7 Troubleshooting › 7.3 Incident-Liste · Tabelle 9: Incidents der Zent,"CAASUP-0351,DDSUP-0794,DDSUP-1201,ESSUP-"
5,knn,6,2bb722f565a0-0046,0.7441,14,text,5.3 Standardabläufe › 5.3.2 Client-Secret rotieren (SOP-ZSD-03),"SOP-ZSD-03,vpp-app-p01,vpp-app-p02"
6,bm25,1,2bb722f565a0-0055,8.4317,17,table,5 Betrieb › 5.5 Standard Operating Procedures · Tabelle 7: Standard Op,"SOP-CAAS-06,SOP-ZSD-01,SOP-ZSD-02,SOP-ZS"
7,bm25,2,2bb722f565a0-0062,6.1493,19,text,6 Abhängigkeiten und Auswirkungen › 6.3 Zirkuläre Abhängigkeit und ihr,"BHB-PLT-0001,SOP-CAAS-06,SOP-ZSD-06"
8,bm25,3,2bb722f565a0-0086,5.9080,26,table,"9 Rollen, Verantwortlichkeiten und Eskalation › 9.1 Rollen und Erreich","FW-ZSD-001,FW-ZSD-015,SOP-CAAS-06"
9,bm25,4,2bb722f565a0-0012,4.7667,5,table,1 Dokumentinformation › 1.3 Verwandte Dokumente · Tabelle 2: Verwandte,"BHB-PLT-0001,SOP-CAAS-06"


## 8 · Graph: laden, 1 Hop, Belegstellen

Der Chat fragt `…-nodes` nie ab. Er lädt **einmal** `GET /<prefix>-documents/_search {"query": {"match_all": {}}, "_source": {"includes": [… "graph"]}}` (`rag_retrieval.search.fetch_documents`) und baut daraus `GraphStore` im Speicher: Knoten, Kanten, Adjazenz, Label-Index. Je Frage dann ohne OpenSearch:

1. **Startknoten** (`facts.select_start_nodes`): die Knoten hinter den Labels aus der Frage + `node_ids` der besten Chunks aus §7.
2. **1 Hop** (`GraphStore.expand`): jede Kante, deren `source` oder `target` ein Startknoten ist.
3. **Fakten** (`facts.build_facts`): Kante + Labels beider Enden → eine Zeile wie `Kai Ostermann —RESPONSIBLE_FOR→ SOP-ZSD-06 [BHB-PLT-0007 S. 17]`, max. 40.
4. **Belegstellen**: `provenance.chunk_ids` der Fakten → **`POST /<prefix>-chunks/_mget`** – der einzige Zugriff auf OpenSearch in diesem Schritt; die Chunks werden die 5. Liste der Fusion.

In [11]:
from rag_retrieval import facts as facts_mod
from rag_retrieval.fusion import rrf, to_chunk_hits

st_ = graph.stats()
print(f"fetch_documents → {len(docs)} Handbuch/Handbücher · GraphStore: {st_.node_ids} Knoten-IDs, {st_.edge_ids} Kanten · Titel: {graph.titles}")

seed_hits = to_chunk_hits(rrf(lists), titles=graph.titles)[:5]                    # beste Chunks der vier Kanäle als Saat
start_nodes = facts_mod.select_start_nodes(graph, label_nodes=plan_q.start_nodes, seed_hits=seed_hits, max_nodes=30,
                                           partial_nodes=plan_q.partial_nodes, question_keys=plan_q.question_keys, boost_types=facts_mod.type_boost(FRAGE))
expanded = graph.expand(start_nodes)                                               # edge_id -> Vorkommen (RAM, kein OpenSearch)
facts = facts_mod.build_facts(graph, expanded, start_nodes=start_nodes, label_nodes=plan_q.start_nodes,
                              relation_labels=facts_mod.load_relation_labels(ONTOLOGY_PATH), max_facts=40, boost=facts_mod.relation_boost(FRAGE))
print(f"Startknoten ({len(start_nodes)}): {[graph.label(n) for n in start_nodes[:8]]} …\n1 Hop: {len(expanded)} Kanten → {len(facts)} Fakten, die ersten 6:")
for f in facts[:6]: print("  -", f.rendered[:170])

reached = [nid for f in facts for nid in (f.source_id, f.target_id) if nid not in start_nodes]
graph_chunk_ids = facts_mod.graph_chunk_ids(facts, graph, label_nodes=plan_q.start_nodes, neighbours=reached, limit=15)
st, mg = req("POST", f"/{NAMES.chunks}/_mget", body={"ids": graph_chunk_ids}, params={"_source_excludes": "embedding,bboxes"})
fetched = {d_["_id"]: d_["_source"] for d_ in mg["docs"] if d_.get("found")}
lists["graph"] = [RawHit(chunk_id=cid, score=None, source=fetched[cid]) for cid in graph_chunk_ids if cid in fetched]
print(f"\nPOST /{NAMES.chunks}/_mget mit {len(graph_chunk_ids)} IDs → {len(fetched)} gefunden: {graph_chunk_ids[:6]} …")

fetch_documents → 1 Handbuch/Handbücher · GraphStore: 250 Knoten-IDs, 154 Kanten · Titel: {'BHB-PLT-0007': 'Betriebshandbuch ZSD - Zentrale Sicherheitsdienste'}
Startknoten (30): ['SOP-ZSD-06', 'SOP-ZSD-01', 'Client-Secret rotieren (SOP-ZSD-03)', 'SOP-ZSD-05', 'SOP-ZSD-02', 'SOP-CAAS-06', 'ZSD - Zentrale Sicherheitsdienste', 'FW-ZSD-004'] …
1 Hop: 61 Kanten → 40 Fakten, die ersten 6:
  - Kai Ostermann —RESPONSIBLE_FOR (verantwortlich für)→ SOP-ZSD-06 (raci=verantwortlich, is_deputy=False) — „SOP-ZSD-06 | Kaltstart der Sicherheitsdienste | Notfall, Stromab
  - Sabine Wollmer —RESPONSIBLE_FOR (verantwortlich für)→ SOP-ZSD-01 (raci=verantwortlich, is_deputy=False) — „SOP-ZSD-01 | Zertifikatsantrag bearbeiten, ausstellen, sperren 
  - Kai Ostermann —RESPONSIBLE_FOR (verantwortlich für)→ Client-Secret rotieren (SOP-ZSD-03) (raci=verantwortlich, is_deputy=False) — „SOP-ZSD-03 | Client-Secret rotieren | J
  - Marcel Ebert —RESPONSIBLE_FOR (verantwortlich für)→ SOP-ZSD-05 (raci=verantwortlich,


POST /nbtest-chunks/_mget mit 15 IDs → 15 gefunden: ['2bb722f565a0-0055', '2bb722f565a0-0088', '2bb722f565a0-0007', '2bb722f565a0-0001', '2bb722f565a0-0002', '2bb722f565a0-0003'] …


## 9 · RRF: von fünf Listen zur Endliste

`rag_retrieval.fusion.rrf`: **`score(chunk) = Σ 1 / (60 + Rang)`** über jede Liste, in der der Chunk vorkommt. Nur Ränge, nie die OpenSearch-Scores. Gleichstand: mehr Kanäle zuerst, dann besserer bester Rang, dann `chunk_id`. Erst die vier Suchkanäle, dann mit der Graph-Liste als fünfter; Endliste = `FINAL_K`.

In [12]:
def rrf_tabelle(lists, n):
    fused = rrf(lists)
    rows = []
    for i, f in enumerate(fused[:n], start=1):
        row = {"Rang": i, "chunk_id": f.chunk_id, "RRF": round(f.fused_score, 4), "Kanäle": len(f.channels)}
        for ch in lists: row[ch] = next((c.rank for c in f.channels if c.channel == ch), "–")
        s = f.source; row["Seite"] = ",".join(map(str, s.get("page_numbers", []))); row["Wo"] = (" › ".join(s.get("heading_breadcrumb", [])[-2:]) + (" · " + s["caption"] if s.get("caption") else ""))[:60]
        rows.append(row)
    return pd.DataFrame(rows)

vier = {ch: l for ch, l in lists.items() if ch != "graph"}
display(Markdown(f"**Vier Suchkanäle** (Top {FINAL_K}; Spalten kNN/BM25/… = Rang in dieser Liste, – = nicht gefunden):")); display(rrf_tabelle(vier, FINAL_K))
display(Markdown("**Mit Graph-Liste** (die Endliste des Chats, vor dem Nachrücken der 2 besten Graph-Belege):")); display(rrf_tabelle(lists, FINAL_K))
top = rrf(vier)[0]
print("Rechenbeispiel Rang 1:", top.chunk_id, "=", " + ".join(f"1/(60+{c.rank})" for c in top.channels), "=", round(top.fused_score, 4))

**Vier Suchkanäle** (Top 10; Spalten kNN/BM25/… = Rang in dieser Liste, – = nicht gefunden):

,Rang,chunk_id,RRF,Kanäle,knn,bm25,identifier,label,Seite,Wo
0,1,2bb722f565a0-0055,0.0656,4,1,1,1,1,17,5 Betrieb › 5.5 Standard Operating Procedures · Tabelle 7: S
1,2,2bb722f565a0-0062,0.0460,3,–,2,2,13,19,6 Abhängigkeiten und Auswirkungen › 6.3 Zirkuläre Abhängigke
2,3,2bb722f565a0-0050,0.0449,3,2,8,–,11,15,5 Betrieb › 5.4 Zertifikatslebenszyklus
3,4,2bb722f565a0-0044,0.0446,3,4,10,–,8,"13,14",5.3 Standardabläufe › 5.3.1 Keycloak rollierend neu starten
4,5,2bb722f565a0-0009,0.0439,3,8,15,–,3,4,1 Dokumentinformation › 1.2 Änderungshistorie · Tabelle 1: Ä
5,6,2bb722f565a0-0075,0.0411,3,7,18,–,15,23,7 Troubleshooting › 7.3 Incident-Liste · Tabelle 9: Incident
6,7,2bb722f565a0-0049,0.0401,3,15,20,–,10,15,5.3 Standardabläufe › 5.3.3 Vault entsiegeln (SOP-ZSD-05)
7,8,2bb722f565a0-0002,0.0308,2,3,7,–,–,1,ZSD - Zentrale Sicherheits dienste
8,9,2bb722f565a0-0086,0.0302,2,10,3,–,–,26,"9 Rollen, Verantwortlichkeiten und Eskalation › 9.1 Rollen u"
9,10,2bb722f565a0-0046,0.0296,2,6,–,–,9,14,5.3 Standardabläufe › 5.3.2 Client-Secret rotieren (SOP-ZSD-


**Mit Graph-Liste** (die Endliste des Chats, vor dem Nachrücken der 2 besten Graph-Belege):

,Rang,chunk_id,RRF,Kanäle,knn,bm25,identifier,label,graph,Seite,Wo
0,1,2bb722f565a0-0055,0.0820,5,1,1,1,1,1,17,5 Betrieb › 5.5 Standard Operating Procedures · Tabelle 7: S
1,2,2bb722f565a0-0062,0.0609,4,–,2,2,13,7,19,6 Abhängigkeiten und Auswirkungen › 6.3 Zirkuläre Abhängigke
2,3,2bb722f565a0-0044,0.0581,4,4,10,–,8,14,"13,14",5.3 Standardabläufe › 5.3.1 Keycloak rollierend neu starten
3,4,2bb722f565a0-0002,0.0462,3,3,7,–,–,5,1,ZSD - Zentrale Sicherheits dienste
4,5,2bb722f565a0-0050,0.0449,3,2,8,–,11,–,15,5 Betrieb › 5.4 Zertifikatslebenszyklus
5,6,2bb722f565a0-0007,0.0445,3,11,9,–,–,3,4,"1 Dokumentinformation › 1.1 Zweck, Zielgruppe, Geltungsberei"
6,7,2bb722f565a0-0046,0.0444,3,6,–,–,9,8,14,5.3 Standardabläufe › 5.3.2 Client-Secret rotieren (SOP-ZSD-
7,8,2bb722f565a0-0009,0.0439,3,8,15,–,3,–,4,1 Dokumentinformation › 1.2 Änderungshistorie · Tabelle 1: Ä
8,9,2bb722f565a0-0088,0.0438,3,20,6,–,–,2,26,"9 Rollen, Verantwortlichkeiten und Eskalation › 9.2 Eskalati"
9,10,2bb722f565a0-0071,0.0421,3,–,11,–,14,9,22,7.2 Dokumentierte Störungsbilder › 7.2.4 Vault nach Knotenwa


Rechenbeispiel Rang 1: 2bb722f565a0-0055 = 1/(60+1) + 1/(60+1) + 1/(60+1) + 1/(60+1) = 0.0656


## 10 · Aufräumen und Ergebnis

Bestanden heißt: §2 verbindet, §3 zeigt `opensearch-knn` und die deutschen Tokens, §4 `OK` beim Roundtrip, §5 vier Indizes `green`/`yellow` mit Alias, §6 `active` mit den erwarteten Zählungen, §7 vier Kanäle mit Treffern, §8 Fakten + `mget`, §9 eine Endliste, in der die SOP-Tabelle oben steht. Läuft das gegen den lokalen Container **und** mit `PROFIL = "remote"`, verhält sich die entfernte Instanz für unsere Zwecke gleich.

`AUFRAEUMEN = True` löscht alle `<PREFIX>-*`-Indizes und die Pipeline (`DELETE /<index>`, `DELETE /_search/pipeline/<prefix>-rrf`). Die produktiven `bhb-*` bleiben unberührt.

In [13]:
if AUFRAEUMEN:
    st, cat = req("GET", f"/_cat/indices/{PREFIX}-*", params={"format": "json", "h": "index"})
    for row in cat or []:
        print("DELETE /" + row["index"], "→", req("DELETE", "/" + row["index"])[0])
    print(f"DELETE /_search/pipeline/{NAMES.rrf_pipeline} →", req("DELETE", f"/_search/pipeline/{NAMES.rrf_pipeline}")[0])
else:
    st, cat = req("GET", f"/_cat/indices/{PREFIX}-*", params={"format": "json", "h": "index,docs.count"})
    print("Indizes bleiben (AUFRAEUMEN = False):"); display(pd.DataFrame(cat))

Indizes bleiben (AUFRAEUMEN = False):


,index,docs.count
0,nbtest-nodes-v1,250
1,nbtest-manifest-v1,1
2,nbtest-chunks-v1,112
3,nbtest-documents-v1,1
